# AI工学101 — 第16回

## scikit-learnの前処理パイプライン：Pipelineで「学習フロー」を安全に組み立てる

前回は分類モデルの評価として、

* Confusion Matrix（混同行列）
* Precision and Recall（適合率・再現率）
* F1 score（F1スコア）

を学びました。

今回は実務で非常によく使われる

> **scikit-learn の `Pipeline`**

を学びます。

実は、ここまで毎回書いてきた

```python
StandardScaler
↓

モデル作成
↓

fit
↓

predict
```

という流れを一つにまとめるための仕組みです。

---

# 🎯 今日のゴール

今日の授業では、

* Pipelineが必要な理由を理解する
* 前処理とモデルを一つにまとめる
* データリークを防ぐ考え方を身につける
* 今後のモデル比較を簡単にできるようになる

---

# 📖 講義（約20分）

## なぜPipelineが必要？

これまで毎回こんなコードを書いていました。

```python
scaler = StandardScaler()

scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)

X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()

model.fit(
    X_train_scaled,
    y_train
)

pred = model.predict(
    X_test_scaled
)
```

動きます。

でも問題があります。

* transformを忘れる
* testでfitしてしまう
* 前処理をモデルごとに書き直す

などのミスが起きやすくなります。

そこで登場するのが

```
Pipeline
```

です。

---

# 📖 Pipelineとは？

Pipelineは

```
前処理①

↓

前処理②

↓

モデル
```

という流れを

一つのオブジェクト

として扱えます。

つまり

```
データ

↓

Pipeline

↓

予測
```

になります。

---

# 💻 実習1：データ準備

```python
import numpy as np

X = np.array([
    [160,50],
    [165,55],
    [170,60],
    [175,68],
    [180,75],
    [185,82],
    [190,90],
    [195,98]
])

y = np.array([
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    1
])
```

---

train/test分割。

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=42,
    test_size=0.25
)
```

---

# 💻 実習2：Pipelineを作る

読み込み。

```python
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
```

作成。

```python
pipe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression()
    )
])
```

これだけです。

---

# 💻 実習3：学習

普通に

```python
pipe.fit(
    X_train,
    y_train
)
```

するだけ。

内部では

```
fit scaler

↓

transform

↓

fit model
```

が自動で行われています。

---

# 💻 実習4：予測

```python
pred = pipe.predict(
    X_test
)

print(pred)
```

さらに

```python
prob = pipe.predict_proba(
    X_test
)

print(prob)
```

も使えます。

Pipelineなのに

モデルと同じように扱えます。

---

# 📖 Pipelineの中で何が起きている？

イメージはこうです。

```text
X_train

↓

StandardScaler.fit()

↓

StandardScaler.transform()

↓

LogisticRegression.fit()
```

予測では、

```text
X_test

↓

StandardScaler.transform()

↓

LogisticRegression.predict()
```

となります。

ここで重要なのは、

> **テストデータでは `fit()` が実行されない**

ことです。

つまり、

データリークを自然に防げます。

---

# 💻 実習5：Accuracyを計算

```python
from sklearn.metrics import accuracy_score

pred = pipe.predict(
    X_test
)

print(
    accuracy_score(
        y_test,
        pred
    )
)
```

今までと全く同じです。

---

# 💻 実習6：モデル交換

ここがPipeline最大の利点。

例えば

```python
from sklearn.tree import DecisionTreeClassifier
```

を使うなら、

ここだけ変更。

```python
pipe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        DecisionTreeClassifier()
    )
])
```

評価コードは

**一切変更不要。**

実務では

この書き方が非常に多いです。

---

# 💻 実習7：Pipelineの中を見る

Pipelineの中身は

```python
print(pipe.named_steps)
```

で確認できます。

例えば

```python
pipe.named_steps["scaler"]
```

なら

StandardScaler。

```python
pipe.named_steps["model"]
```

なら

LogisticRegression

です。

---

# ✍️ 演習

今日のデータ。

```python
X = np.array([
    [20,150],
    [25,160],
    [30,170],
    [35,180],
    [40,185],
    [45,190],
    [50,195],
    [55,200]
])

y = np.array([
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    1
])
```

---

## 問1

train/test分割してください。

---

## 問2

Pipelineを作ってください。

中身は

* StandardScaler
* LogisticRegression

です。

---

## 問3

学習してください。

---

## 問4

予測してください。

---

## 問5

Accuracyを表示してください。

---

## 問6（ボス戦👾）

`pipe.named_steps`

を使って、

* StandardScaler
* LogisticRegression

を取得し、

表示してください。

---

# 🌿 今日のまとめ

今日は、実務でほぼ必須となる `Pipeline` を学びました。

Pipelineを使うことで、

```text
生データ
      ↓
StandardScaler
      ↓
LogisticRegression
      ↓
予測
```

という一連の流れを、安全かつ再利用しやすい形でまとめられます。

ここまで学んだ内容を振り返ると、

```text
NumPy
      ↓
前処理
      ↓
StandardScaler
      ↓
train_test_split
      ↓
LinearRegression
      ↓
LogisticRegression
      ↓
評価指標
      ↓
Pipeline ← ★今日
```

と、機械学習プロジェクトの基本的な流れが一本につながってきました。

---

# 🔜 第17回予告

次回は

> **特徴量（Feature Engineering）**

です。

テーマは、

* 特徴量とは何か
* 不要な特徴量がモデルに与える影響
* 特徴量選択の考え方
* 多項式特徴量（Polynomial Features）の基礎

です。

ここでは「AIに何を入力するか」が性能を大きく左右することを学びます。実務でも非常に重要な工程であり、「モデルを変える前に特徴量を見直す」という発想の土台になります。